In [1]:
import numpy as np
import kwant
import scipy
import scipy.sparse.linalg as sla
import numpy.linalg as LA
from scipy.constants import hbar, m_e, pi, eV
meV = 1e-3 * eV

C:\Users\guanzhongwang\Anaconda3\envs\cps-theory\lib\site-packages\kwant\solvers\default.py:16: RuntimeWarning: MUMPS is not available, SciPy built-in solver will be used as a fallback. Performance can be very poor in this case.
  warnings.warn("MUMPS is not available, "


In [2]:
sigma_0 = np.array([[1, 0], [0, 1]])
sigma_x = np.array([[0, 1], [1, 0]])
sigma_y = np.array([[0, -1j], [1j, 0]])
sigma_z = np.array([[1, 0], [0, -1]])

tau_0 = np.array([[1, 0], [0, 1]])
tau_x = np.array([[0, 1], [1, 0]])
tau_y = np.array([[0, -1j], [1j, 0]])
tau_z = np.array([[1, 0], [0, -1]])

In [3]:
def make_hybrid_nanowire(length=300, a=10):
    def wire(pos):
        x = pos[0]
        return 0 <= x <= length
    def h1(site, t, mu, Delta, Ez, Ux, x_range):
        x = site.pos[0]
        i = int(x/a)
        return (2*t - mu + Ux[i]) * np.kron(tau_z, sigma_0) + Delta * np.kron(tau_y, sigma_y) \
            + Ez * np.kron(tau_z, sigma_z)
    def h2(site1, site2, t, alpha, theta):
        return -t * np.kron(tau_z, sigma_0) \
            - 1j * alpha * (np.cos(theta) * np.kron(tau_z, sigma_y) + np.sin(theta) * np.kron(tau_0, sigma_z))
    lat = kwant.lattice.chain(a, norbs=4)
    syst = kwant.Builder()
    syst[lat.shape(wire, [0])] = h1
    syst[lat.neighbors()] = h2
    return syst.finalized()

In [4]:
def get_matrices(ham_params, N=4):
    H = hybrid_nanowire.hamiltonian_submatrix(params=ham_params, sparse=True)
    es, wfs = sla.eigsh(A=H, k=N, which='LM', sigma=0.0, return_eigenvectors=True)
    
    a_su_su, a_su_sd, a_sd_su, a_sd_sd = (0, 0, 0, 0)
    b_su_su, b_su_sd, b_sd_su, b_sd_sd = (0, 0, 0, 0)
    
    for i, E_i in enumerate(es):
        E_abs = E_i / ham_params['Delta']
        if E_abs > 0:
            a_su_su += (wfs[0, i] * wfs[-2, i].conj() - wfs[-4, i] * wfs[2, i].conj()) / E_abs
            a_su_sd += (wfs[0, i] * wfs[-1, i].conj() - wfs[-3, i] * wfs[2, i].conj()) / E_abs
            a_sd_su += (wfs[1, i] * wfs[-2, i].conj() - wfs[-4, i] * wfs[3, i].conj()) / E_abs
            a_sd_sd += (wfs[1, i] * wfs[-1, i].conj() - wfs[-3, i] * wfs[3, i].conj()) / E_abs
            
            b_su_su += (wfs[0, i] * wfs[-4, i].conj() - wfs[-2, i] * wfs[2, i].conj()) / E_abs
            b_su_sd += (wfs[0, i] * wfs[-3, i].conj() - wfs[-1, i] * wfs[2, i].conj()) / E_abs
            b_sd_su += (wfs[1, i] * wfs[-4, i].conj() - wfs[-2, i] * wfs[3, i].conj()) / E_abs
            b_sd_sd += (wfs[1, i] * wfs[-3, i].conj() - wfs[-1, i] * wfs[3, i].conj()) / E_abs
            
    a_matrix = np.asarray([[a_su_su, a_su_sd], [a_sd_su, a_sd_sd]])
    b_matrix = np.asarray([[b_su_su, b_su_sd], [b_sd_su, b_sd_sd]])
    return a_matrix, b_matrix

# 1. angle

In [5]:
geo_params = dict(length=200, a=10)
hybrid_nanowire = make_hybrid_nanowire(**geo_params)

np.random.seed(2)
x_range = np.linspace(0, 200, 21)
U_D = 10
alpha_R = 0.15
ham_params = dict(
    t=25.4,
    alpha=alpha_R*5,
    mu=6.3,
    x_range=x_range,
    Ux=U_D * (2*np.random.rand(len(x_range)) - 1),
    Delta=0.15,
    Ez=0.06 + 1e-5,
    theta=0*np.pi/2
)

In [6]:
num_of_theta = 100
theta_range = np.linspace(0, 2*np.pi, num_of_theta)

CAR_uu_vs_theta = np.zeros(len(theta_range))
CAR_ud_vs_theta = np.zeros(len(theta_range))
CAR_du_vs_theta = np.zeros(len(theta_range))
CAR_dd_vs_theta = np.zeros(len(theta_range))

ECT_uu_vs_theta = np.zeros(len(theta_range))
ECT_ud_vs_theta = np.zeros(len(theta_range))
ECT_du_vs_theta = np.zeros(len(theta_range))
ECT_dd_vs_theta = np.zeros(len(theta_range))

In [7]:
for i, ham_params['theta'] in enumerate(theta_range):
    a_matrix, b_matrix = get_matrices(ham_params=ham_params, N=40)
    CAR_uu_vs_theta[i] = abs(a_matrix[0, 0])**2
    CAR_ud_vs_theta[i] = abs(a_matrix[0, 1])**2
    CAR_du_vs_theta[i] = abs(a_matrix[1, 0])**2
    CAR_dd_vs_theta[i] = abs(a_matrix[1, 1])**2
    
    ECT_uu_vs_theta[i] = abs(b_matrix[0, 0])**2
    ECT_ud_vs_theta[i] = abs(b_matrix[0, 1])**2
    ECT_du_vs_theta[i] = abs(b_matrix[1, 0])**2
    ECT_dd_vs_theta[i] = abs(b_matrix[1, 1])**2

In [8]:
np.savez('./raw_data/theory_P_vs_theta.npz', 
         theta_range=theta_range,
         CAR_uu_vs_theta=CAR_uu_vs_theta,
         CAR_ud_vs_theta=CAR_ud_vs_theta,
         CAR_du_vs_theta=CAR_du_vs_theta,
         CAR_dd_vs_theta=CAR_dd_vs_theta,
         ECT_uu_vs_theta=ECT_uu_vs_theta,
         ECT_ud_vs_theta=ECT_ud_vs_theta,
         ECT_du_vs_theta=ECT_du_vs_theta,
         ECT_dd_vs_theta=ECT_dd_vs_theta)

# 2. ratio vs. mu

In [9]:
geo_params = dict(length=200, a=10)
hybrid_nanowire = make_hybrid_nanowire(**geo_params)

np.random.seed(2)
x_range = np.linspace(0, 200, 21)
U_D = 10
alpha_R = 0.15
ham_params = dict(
    t=25.4,
    alpha=alpha_R*5,
    mu=6.3,
    x_range=x_range,
    Ux=U_D * (2*np.random.rand(len(x_range)) - 1),
    Delta=0.15,
    Ez=0.06 + 1e-5,
    theta=0*np.pi/2
)

theta_range = np.linspace(0, np.pi, 50)
mu_range = np.linspace(-0.4, 0.4, 30)

In [10]:
mu1 = 3.784 #, 6.128, 9.784
CAR_uu_avg_vs_mu = np.zeros(len(mu_range))
CAR_ud_avg_vs_mu = np.zeros(len(mu_range))

for i, mu in enumerate(mu_range):
    ham_params['mu'] = mu + mu1
    CAR_uu_vs_theta = np.zeros(len(theta_range))
    CAR_ud_vs_theta = np.zeros(len(theta_range))
    for j, ham_params['theta'] in enumerate(theta_range):
        a_matrix, b_matrix = get_matrices(ham_params=ham_params, N=20)
        CAR_uu_vs_theta[j] = abs(a_matrix[0, 0])**2
        CAR_ud_vs_theta[j] = abs(a_matrix[0, 1])**2
    CAR_uu_avg_vs_mu[i] = np.mean(CAR_uu_vs_theta)
    CAR_ud_avg_vs_mu[i] = np.mean(CAR_ud_vs_theta)
ratio_vs_mu1 = CAR_uu_avg_vs_mu / CAR_ud_avg_vs_mu

In [11]:
mu2 = 6.128 #, 9.784
CAR_uu_avg_vs_mu = np.zeros(len(mu_range))
CAR_ud_avg_vs_mu = np.zeros(len(mu_range))

for i, mu in enumerate(mu_range):
    ham_params['mu'] = mu + mu2
    CAR_uu_vs_theta = np.zeros(len(theta_range))
    CAR_ud_vs_theta = np.zeros(len(theta_range))
    for j, ham_params['theta'] in enumerate(theta_range):
        a_matrix, b_matrix = get_matrices(ham_params=ham_params, N=20)
        CAR_uu_vs_theta[j] = abs(a_matrix[0, 0])**2
        CAR_ud_vs_theta[j] = abs(a_matrix[0, 1])**2
    CAR_uu_avg_vs_mu[i] = np.mean(CAR_uu_vs_theta)
    CAR_ud_avg_vs_mu[i] = np.mean(CAR_ud_vs_theta)
ratio_vs_mu2 = CAR_uu_avg_vs_mu / CAR_ud_avg_vs_mu

In [12]:
mu3 = 9.784
CAR_uu_avg_vs_mu = np.zeros(len(mu_range))
CAR_ud_avg_vs_mu = np.zeros(len(mu_range))

for i, mu in enumerate(mu_range):
    ham_params['mu'] = mu + mu3
    CAR_uu_vs_theta = np.zeros(len(theta_range))
    CAR_ud_vs_theta = np.zeros(len(theta_range))
    for j, ham_params['theta'] in enumerate(theta_range):
        a_matrix, b_matrix = get_matrices(ham_params=ham_params, N=20)
        CAR_uu_vs_theta[j] = abs(a_matrix[0, 0])**2
        CAR_ud_vs_theta[j] = abs(a_matrix[0, 1])**2
    CAR_uu_avg_vs_mu[i] = np.mean(CAR_uu_vs_theta)
    CAR_ud_avg_vs_mu[i] = np.mean(CAR_ud_vs_theta)
ratio_vs_mu3 = CAR_uu_avg_vs_mu / CAR_ud_avg_vs_mu

In [13]:
np.savez('./raw_data/theory_ratio_vs_mu.npz',
         dmu_range=mu_range,
         mu1=mu1,
         mu2=mu2,
         mu3=mu3,
         ratio_vs_mu1=ratio_vs_mu1,
         ratio_vs_mu2=ratio_vs_mu2,
         ratio_vs_mu3=ratio_vs_mu3)

# 3. ratio vs. alpha

## Short wire

In [14]:
L = 200 # 200, 350
mu1, mu2, mu3 = 3.784, 6.128, 9.784

geo_params = dict(length=L, a=10) # L = 200 or 350
hybrid_nanowire = make_hybrid_nanowire(**geo_params)

np.random.seed(2)
x_range = np.linspace(0, L, int(L/10+1)) 
U_D = 10
ham_params = dict(
    t=25.4,
    alpha=0,
    mu=mu1, 
    x_range=x_range,
    Ux=U_D * (2*np.random.rand(len(x_range)) - 1),
    Delta=0.15,
    Ez=0.06 + 1e-5,
    theta=0*np.pi/2
)

alpha_range = np.linspace(0, 0.5, 50)
theta_range = np.linspace(0, np.pi, 50)

In [15]:
ham_params['mu'] = mu1

CAR_uu_avg_vs_alpha = np.zeros(len(alpha_range))
CAR_ud_avg_vs_alpha = np.zeros(len(alpha_range))

for i, alpha_R in enumerate(alpha_range):
    ham_params['alpha'] = alpha_R * 5
    CAR_uu_vs_theta = np.zeros(len(theta_range))
    CAR_ud_vs_theta = np.zeros(len(theta_range))
    for j, ham_params['theta'] in enumerate(theta_range):
        a_matrix, b_matrix = get_matrices(ham_params=ham_params, N=20)
        CAR_uu_vs_theta[j] = abs(a_matrix[0, 0])**2
        CAR_ud_vs_theta[j] = abs(a_matrix[0, 1])**2
    CAR_uu_avg_vs_alpha[i] = np.mean(CAR_uu_vs_theta)
    CAR_ud_avg_vs_alpha[i] = np.mean(CAR_ud_vs_theta)
    
ratio_vs_alpha1 = CAR_uu_avg_vs_alpha / CAR_ud_avg_vs_alpha

In [16]:
ham_params['mu'] = mu2

CAR_uu_avg_vs_alpha = np.zeros(len(alpha_range))
CAR_ud_avg_vs_alpha = np.zeros(len(alpha_range))

for i, alpha_R in enumerate(alpha_range):
    ham_params['alpha'] = alpha_R * 5
    CAR_uu_vs_theta = np.zeros(len(theta_range))
    CAR_ud_vs_theta = np.zeros(len(theta_range))
    for j, ham_params['theta'] in enumerate(theta_range):
        a_matrix, b_matrix = get_matrices(ham_params=ham_params, N=20)
        CAR_uu_vs_theta[j] = abs(a_matrix[0, 0])**2
        CAR_ud_vs_theta[j] = abs(a_matrix[0, 1])**2
    CAR_uu_avg_vs_alpha[i] = np.mean(CAR_uu_vs_theta)
    CAR_ud_avg_vs_alpha[i] = np.mean(CAR_ud_vs_theta)
    
ratio_vs_alpha2 = CAR_uu_avg_vs_alpha / CAR_ud_avg_vs_alpha

In [17]:
ham_params['mu'] = mu3

CAR_uu_avg_vs_alpha = np.zeros(len(alpha_range))
CAR_ud_avg_vs_alpha = np.zeros(len(alpha_range))

for i, alpha_R in enumerate(alpha_range):
    ham_params['alpha'] = alpha_R * 5
    CAR_uu_vs_theta = np.zeros(len(theta_range))
    CAR_ud_vs_theta = np.zeros(len(theta_range))
    for j, ham_params['theta'] in enumerate(theta_range):
        a_matrix, b_matrix = get_matrices(ham_params=ham_params, N=20)
        CAR_uu_vs_theta[j] = abs(a_matrix[0, 0])**2
        CAR_ud_vs_theta[j] = abs(a_matrix[0, 1])**2
    CAR_uu_avg_vs_alpha[i] = np.mean(CAR_uu_vs_theta)
    CAR_ud_avg_vs_alpha[i] = np.mean(CAR_ud_vs_theta)
    
ratio_vs_alpha3 = CAR_uu_avg_vs_alpha / CAR_ud_avg_vs_alpha

In [18]:
np.savez('./raw_data/theory_ratio_vs_SOC_short.npz',
         alpha_range = alpha_range,
         ratio_vs_alpha_1 = ratio_vs_alpha1,
         ratio_vs_alpha_2 = ratio_vs_alpha2,
         ratio_vs_alpha_3 = ratio_vs_alpha3,
         mu1 = mu1,
         mu2 = mu2,
         mu3 = mu3)

## Long wire

In [19]:
L = 350
mu1, mu2, mu3 = 2.742, 4.131, 6.128

geo_params = dict(length=L, a=10) # L = 200 or 350
hybrid_nanowire = make_hybrid_nanowire(**geo_params)

np.random.seed(2)
x_range = np.linspace(0, L, int(L/10+1)) 
U_D = 10
ham_params = dict(
    t=25.4,
    alpha=0,
    mu=mu1, 
    x_range=x_range,
    Ux=U_D * (2*np.random.rand(len(x_range)) - 1),
    Delta=0.15,
    Ez=0.06 + 1e-5,
    theta=0*np.pi/2
)

alpha_range = np.linspace(0, 0.5, 50)
theta_range = np.linspace(0, np.pi, 50)

In [20]:
ham_params['mu'] = mu1

CAR_uu_avg_vs_alpha = np.zeros(len(alpha_range))
CAR_ud_avg_vs_alpha = np.zeros(len(alpha_range))

for i, alpha_R in enumerate(alpha_range):
    ham_params['alpha'] = alpha_R * 5
    CAR_uu_vs_theta = np.zeros(len(theta_range))
    CAR_ud_vs_theta = np.zeros(len(theta_range))
    for j, ham_params['theta'] in enumerate(theta_range):
        a_matrix, b_matrix = get_matrices(ham_params=ham_params, N=20)
        CAR_uu_vs_theta[j] = abs(a_matrix[0, 0])**2
        CAR_ud_vs_theta[j] = abs(a_matrix[0, 1])**2
    CAR_uu_avg_vs_alpha[i] = np.mean(CAR_uu_vs_theta)
    CAR_ud_avg_vs_alpha[i] = np.mean(CAR_ud_vs_theta)
    
ratio_vs_alpha1 = CAR_uu_avg_vs_alpha / CAR_ud_avg_vs_alpha

In [21]:
ham_params['mu'] = mu2

CAR_uu_avg_vs_alpha = np.zeros(len(alpha_range))
CAR_ud_avg_vs_alpha = np.zeros(len(alpha_range))

for i, alpha_R in enumerate(alpha_range):
    ham_params['alpha'] = alpha_R * 5
    CAR_uu_vs_theta = np.zeros(len(theta_range))
    CAR_ud_vs_theta = np.zeros(len(theta_range))
    for j, ham_params['theta'] in enumerate(theta_range):
        a_matrix, b_matrix = get_matrices(ham_params=ham_params, N=20)
        CAR_uu_vs_theta[j] = abs(a_matrix[0, 0])**2
        CAR_ud_vs_theta[j] = abs(a_matrix[0, 1])**2
    CAR_uu_avg_vs_alpha[i] = np.mean(CAR_uu_vs_theta)
    CAR_ud_avg_vs_alpha[i] = np.mean(CAR_ud_vs_theta)
    
ratio_vs_alpha2 = CAR_uu_avg_vs_alpha / CAR_ud_avg_vs_alpha

In [ ]:
ham_params['mu'] = mu3

CAR_uu_avg_vs_alpha = np.zeros(len(alpha_range))
CAR_ud_avg_vs_alpha = np.zeros(len(alpha_range))

for i, alpha_R in enumerate(alpha_range):
    ham_params['alpha'] = alpha_R * 5
    CAR_uu_vs_theta = np.zeros(len(theta_range))
    CAR_ud_vs_theta = np.zeros(len(theta_range))
    for j, ham_params['theta'] in enumerate(theta_range):
        a_matrix, b_matrix = get_matrices(ham_params=ham_params, N=20)
        CAR_uu_vs_theta[j] = abs(a_matrix[0, 0])**2
        CAR_ud_vs_theta[j] = abs(a_matrix[0, 1])**2
    CAR_uu_avg_vs_alpha[i] = np.mean(CAR_uu_vs_theta)
    CAR_ud_avg_vs_alpha[i] = np.mean(CAR_ud_vs_theta)
    
ratio_vs_alpha3 = CAR_uu_avg_vs_alpha / CAR_ud_avg_vs_alpha

In [ ]:
np.savez('./raw_data/theory_ratio_vs_SOC_long.npz',
         alpha_range = alpha_range,
         ratio_vs_alpha_1 = ratio_vs_alpha1,
         ratio_vs_alpha_2 = ratio_vs_alpha2,
         ratio_vs_alpha_3 = ratio_vs_alpha3,
         mu1 = mu1,
         mu2 = mu2,
         mu3 = mu3)